In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.segment_to_graph.models.dataset import VBranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = VBranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, resize_to=1024)

Found 215 branch digraphs...


Processing...
Processing dataset: 0it [00:00, ?it/s]
Done!
Preloading dataset:   0%|          | 0/215 [00:00<?, ?it/s]

## Graph Augment


In [ ]:
ID = 0
graph = dataset.graphs[ID]
fundus = FundusData(image=dataset.fundus_paths[ID])

In [5]:
m, digraph, _ = dataset.draw_jppype(200, augment=True, test=True)
m

[ WARN:0@0.333] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [6]:
digraph, fundus_img = dataset.get_sample(200, augment=True)

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/graph_simplification.py:724: UserWarning: The nodes of branch(es) [219] are superposed,their tips tangent will be set to (0, 0).
  [gdata.tip_tangent(b, d, attr=tangent_key) for b, d in zip(endp_branch, endp_first_tip, strict=True)]
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/graph_simplification.py:822: UserWarning: The nodes of branch(es) [219] are superposed,their tips tangent will be set to (0, 0).
  tips_tan = torch.from_numpy(geodata.tip_tangent(attr=tangent))


In [7]:
import numpy as np

np.argmax([False, False, False, False, False])

np.int64(0)

In [8]:
from fundus_vessels_toolkit.utils.fundus_projections import ElasticProjection, IdentityProjection

digraph, fundus_img = dataset.get_sample(0)

%timeit dataset.get_sample(0, augment=True)

fundus_img = fundus_img.transpose(1, 2, 0)  # C,H,W -> H,W,C
shape = fundus_img.shape[0], fundus_img.shape[1]

elastic = ElasticProjection.random(shape, displacement_std=120, smoothing_size=200)
identity = IdentityProjection()
digraph.graph.transform(elastic, inplace=False)
%timeit digraph.graph.transform(elastic, inplace=False)
%timeit elastic.warp(fundus_img, warped_domain="same")

/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/graph_simplification.py:724: UserWarning: The nodes of branch(es) [45] are superposed,their tips tangent will be set to (0, 0).
  [gdata.tip_tangent(b, d, attr=tangent_key) for b, d in zip(endp_branch, endp_first_tip, strict=True)]
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/graph_simplification.py:724: UserWarning: The nodes of branch(es) [160] are superposed,their tips tangent will be set to (0, 0).
  [gdata.tip_tangent(b, d, attr=tangent_key) for b, d in zip(endp_branch, endp_first_tip, strict=True)]
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to_graph/graph_simplification.py:822: UserWarning: The nodes of branch(es) [ 45 160] are superposed,their tips tangent will be set to (0, 0).
  tips_tan = torch.from_numpy(geodata.tip_tangent(attr=tangent))
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/segment_to

90.1 ms ± 1.38 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
14.9 ms ± 630 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
23.2 ms ± 357 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
len(digraph.graph.geometric_data().branch_curve())

235

In [10]:
curve = digraph.graph.branch(0).curve()
elastic = ElasticProjection.random(shape, displacement_std=120, smoothing_size=200)
%timeit elastic.transform(curve)

13.2 μs ± 317 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [11]:
m.views[0].goto(dataset.get_digraph(0).graph.branch(24).midpoint().xy, 3)

AttributeError: 'VBranchDigraphDataset' object has no attribute 'get_digraph'

In [ ]:
for i, data in enumerate(DataLoader(dataset, batch_size=8, num_workers=4)):
    ...